# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset includes records of protein abundance, modifications, and sequences in human samples, derived from extracellular vesicle mass spectrometry analysis.

### Dataset Source
The dataset source is provided as a Croissant schema at:

**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print('Dataset name:', meta.name)
print('Description:', meta.description)
print('\nIdentifier:', getattr(meta, 'identifier', None))
print('Version:', getattr(meta, 'version', None))
print('License:', getattr(meta, 'license', None))


## 2. Data Overview
Review all available record sets and their fields by `@id`. This will guide extraction and analysis in the following steps.

In [ ]:
# Inspect record sets

record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: @id = {rs['@id']}")
    print(f"  Name: {rs.get('name', '<none>')}")
    print(f"  Description: {rs.get('description', '<none>')}")
    print(f"  Fields:")
    for f in rs.get('field', []):
        print(f"    - {f['@id']} (name: {f.get('name', '<none>')})")
    print('-'*55)

# For exploration, choose key record set(s)
# For this dataset, the main record set is expected to be proteins.
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Found Record Set @ids: {record_set_ids}")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. We will display columns and examine the head of the main data table.

*All record sets, fields, and columns are referenced by their `@id` fields only.*

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print('\n---\n')

# Example: Focus on first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    print(f"Fields/columns of primary table (@id={main_record_set_id}):\n", list(main_df.columns))

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping on relevant numeric fields. All column/field references use their `@id` for strict reproducibility.

In [ ]:
# Example: Suppose abundance is field '@id': 'cr:field/abundance'
# We choose numeric_field_id and group_field_id by inspecting the columns above

if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    # Try to infer likely numeric fields
    numeric_candidates = [col for col in main_df.columns if main_df[col].dtype in ['float64', 'int64']]
    print('Numeric column candidates:', numeric_candidates)

    # For this demo, select first numeric column as example
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None

    if numeric_field_id:
        threshold = main_df[numeric_field_id].mean()
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '@id' {main_record_set_id} where {numeric_field_id} > {threshold:.3f} (mean)")
        print(filtered_df.head())

        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, norm_field]].head())

        # Try grouping by first textual/categorical field
        group_candidates = [col for col in main_df.columns if main_df[col].dtype == 'object' and col != numeric_field_id]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version. All references use `@id` variable names for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    if norm_field in filtered_df:
        sns.histplot(filtered_df[norm_field], bins=20, kde=True, ax=ax[1], color='orange')
        ax[1].set_title(f"{numeric_field_id} Normalized (filtered)")
    plt.tight_layout()
    plt.show()

# Optional: Pairwise plot for main numeric fields
if main_record_set_id and numeric_candidates and len(numeric_candidates) > 1:
    sns.pairplot(main_df[numeric_candidates].dropna().sample(min(200, len(main_df))), diag_kind='kde')
    plt.suptitle(f"Pairwise distributions: {main_record_set_id}", y=1.01)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR\^2 dataset, enumerated all record sets by `@id`, extracted main tables, and performed simple EDA using field references by their unique Croissant `@id`.

You can repeat the analysis above for other record sets or experiment with more advanced data processing, all while referencing fields and sets strictly by their Croissant-level `@id`.

**Key learnings:**
- Identified available record sets and major fields (all by `@id`).
- Demonstrated filtering, normalization, and grouping with clear provenance (using `@id`).
- Visualized distributions from the dataset directly loaded via `mlcroissant`.

For more details, visit the [mlcroissant repo](https://github.com/mlcommons/croissant) and refer to [FAIR\^2 dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa).